In [1]:
import sys
from pathlib import Path

API = Path.cwd().parent                     # .../StaffScheduling/api (Jupyter startet im Ordner des Notebooks)
ROOT = API.parent
sys.path[:0] = [str(API), str(API / "tests")]   # app.* und die Test-Bausteine importierbar machen

# --- Datenstrukturen (nur app.domain: braucht pydantic + holidays, keine Datenbank, kein OR-Tools) ---
from app.domain import (
    POLICY,                 # Regelparameter: min_rest_minutes, balance_tolerance_minutes, max_consecutive_nights, ...
    SchedulingDataset,      # die komplette Eingabe eines Monats
    Assignment,             # ein Dienst (Ergebnis): employee_id, date, planning_unit_id, shift_id, staff_level
    Gap,                    # eine Lücke (Ergebnis)
    DemandRequirement, Shift, ShiftType, StaffLevel, AvailabilityType, WishType, FREE_WISHES,
    duty_times,             # echte Start-/Endzeiten eines Dienstes (inkl. Sommerzeit)
    is_working_day,         # Werktag ja/nein
    check_schedule,         # unabhängiger Schedule-Check: Regeln + Kennzahlen
)
from app.domain.rules import BLOCKING_AVAILABILITY, APPROVED_FREE
from app.domain.planning_unit import home_unit_id
import gurobipy as gp
from gurobipy import GRB
from app.domain import SchedulingDataset, PlanningUnitType
from app.domain import POLICY, Assignment, Gap, SchedulingDataset, check_schedule
from app.settings import Settings
from app.solver.models import RunConfiguration, Solution, SolutionStatus, StageReport
from datetime import timedelta
from app.domain import StaffLevel

# --- Testdaten laden ---
from app.solver.bundle import ScheduleInput     # liest input.json der Beispielmonate
import scheduling as fx                         # api/tests/scheduling.py: Bausteine für kleine Testfälle


In [2]:
dataset = ScheduleInput.model_validate_json((ROOT / "plaene" / "2026-04" / "input.json").read_text()).dataset

def only_station(dataset: SchedulingDataset, station_id: int) -> SchedulingDataset:
    """Der Datensatz eingeschränkt auf eine Station: ihr Bedarf und nur die Leute, die dort arbeiten dürfen."""
    # Alle, die an der Station eine Zugehörigkeit haben (Stammpersonal und Springer)
    employees = {m.employee_id for m in dataset.planning_unit_memberships if m.planning_unit_id == station_id}
    # Springerpools bleiben drin: Sie sind die Stammeinheit der Springer, bekommen aber nie Dienste
    pools = {u.planning_unit_id for u in dataset.planning_units if u.type == PlanningUnitType.JUMPER_POOL}
    units = {station_id} | pools

    def keep(rows):
        return tuple(row for row in rows if row.employee_id in employees)

    return SchedulingDataset(
        planning_month=dataset.planning_month,
        planning_units=tuple(u for u in dataset.planning_units if u.planning_unit_id in units),
        shifts=dataset.shifts,
        demand_requirements=tuple(r for r in dataset.demand_requirements if r.planning_unit_id == station_id),
        employees=keep(dataset.employees),
        planning_unit_memberships=tuple(m for m in keep(dataset.planning_unit_memberships) if m.planning_unit_id in units),
        availability=keep(dataset.availability),
        wishes=keep(dataset.wishes),
        monthly_work_accounts=keep(dataset.monthly_work_accounts),
        context=dataset.context.model_copy(update={
            "duties": keep(dataset.context.duties),
            "availability": keep(dataset.context.availability),
        }),
    )

dataset = only_station(dataset, 337)

In [3]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def table(rows, drop=()):
    """Eine Liste von Pydantic-Objekten als DataFrame."""
    return pd.DataFrame([row.model_dump(mode="json", exclude=set(drop)) for row in rows])

print(f"Monat: {dataset.planning_month.label} ({len(dataset.planning_month.dates)} Tage)")
print(f"Stationen: {sorted(dataset.station_ids)}")
for name in ["planning_units", "shifts", "employees", "planning_unit_memberships", "demand_requirements",
             "availability", "wishes", "monthly_work_accounts"]:
    print(f"{name:28s} {len(getattr(dataset, name)):5d} Zeilen")
print(f"{'context.duties':28s} {len(dataset.context.duties):5d} Zeilen  ({dataset.context.covered_from} bis {dataset.context.covered_until})")


Monat: 2026-04 (30 Tage)
Stationen: [337]
planning_units                   3 Zeilen
shifts                           4 Zeilen
employees                       20 Zeilen
planning_unit_memberships       39 Zeilen
demand_requirements            150 Zeilen
availability                     0 Zeilen
wishes                           0 Zeilen
monthly_work_accounts           20 Zeilen
context.duties                  56 Zeilen  (2026-03-27 bis 2026-04-30)


In [4]:
display(table(dataset.planning_units))

def hhmm(minute):
    return f"{minute // 60 % 24:02d}:{minute % 60:02d}" + (" (+1)" if minute >= 1440 else "")

shifts = pd.DataFrame([
    {"shift_id": s.shift_id, "code": s.code, "type": s.type.value,
     "start": hhmm(s.start_minute), "end": hhmm(s.end_minute),
     "segments": ", ".join(f"{hhmm(g.start_minute)}-{hhmm(g.end_minute)}" for g in s.segments),
     "net_work_minutes": s.net_work_minutes}
    for s in dataset.shifts
])
display(shifts)


,planning_unit_id,display_name,type
0,68,PE 68,jumper_pool
1,337,PE 337,station
2,408,PE 408,jumper_pool


,shift_id,code,type,start,end,segments,net_work_minutes
0,1113,F,early,05:55,13:25,"05:55-10:00, 10:30-13:25",420
1,1453,Z,intermediate,08:30,14:15,08:30-14:15,345
2,1605,S,late,13:15,21:00,"13:15-16:00, 16:30-21:00",435
3,1690,N,night,20:10,06:10 (+1),"20:10-21:45, 22:00-00:00 (+1), 00:30 (+1)-06:1...",555


In [5]:
employees = table(dataset.employees)
display(employees)
display(employees["staff_level"].value_counts())

memberships = table(dataset.planning_unit_memberships)
display(memberships)
# Wer darf auf welcher Station mit welcher Qualifikation arbeiten?
display(pd.crosstab(memberships["employee_id"], memberships["planning_unit_id"]))


,employee_id,display_name,staff_level
0,918,Ambriz Waltraud,professional
1,835,Ciaccio Josefa,professional
2,7006,Cormier Sigrid,professional
3,843,Flanagan Mirjam,assistant
4,6318,Gamblin Uschi,professional
5,1167,Min Friedhard,professional
6,839,Säuffert Liselotte,assistant
7,8046,Springer Alina,professional
8,8047,Springer Boris,professional
9,8048,Springer Cem,professional


staff_level
professional    16
assistant        3
trainee          1
Name: count, dtype: int64

,planning_unit_id,employee_id,valid_from,valid_until,staff_level,is_home,is_replacement
0,68,918,2024-10-29,2199-12-31,professional,True,False
1,337,918,2025-12-01,2026-07-31,professional,False,True
2,68,835,2023-04-01,2199-12-31,professional,True,False
3,337,835,2025-12-01,2026-07-31,professional,False,True
4,68,7006,2024-01-01,2199-12-31,professional,True,False
5,337,7006,2025-12-01,2026-07-31,professional,False,True
6,68,843,2023-11-01,2199-12-31,assistant,True,False
7,337,843,2025-12-01,2026-07-31,assistant,False,True
8,68,6318,2023-04-15,2199-12-31,professional,True,False
9,337,6318,2025-12-01,2026-07-31,professional,False,True


planning_unit_id,68,337,408
employee_id,,,
835,1,1,0
839,1,1,0
843,1,1,0
918,1,1,0
1167,0,1,0
1218,1,1,0
5844,1,1,0
6259,1,1,0
6318,1,1,0


In [6]:
demand = table(dataset.demand_requirements)
code = {s.shift_id: s.code for s in dataset.shifts}
demand["shift"] = demand["shift_id"].map(code)
display(demand.head(20))

# Raster je Station: Zeilen = Schicht und Qualifikation, Spalten = Tag
for unit, rows in demand.groupby("planning_unit_id"):
    print(f"Station {unit}")
    display(rows.pivot_table(index=["shift", "staff_level"], columns="date",
                             values="required_count", aggfunc="sum", fill_value=0))


,planning_unit_id,date,shift_id,staff_level,required_count,shift
0,337,2026-04-01,1113,professional,1,F
1,337,2026-04-01,1113,trainee,1,F
2,337,2026-04-01,1605,professional,1,S
3,337,2026-04-01,1605,trainee,1,S
4,337,2026-04-01,1690,professional,1,N
5,337,2026-04-02,1113,professional,1,F
6,337,2026-04-02,1113,trainee,1,F
7,337,2026-04-02,1605,professional,1,S
8,337,2026-04-02,1605,trainee,1,S
9,337,2026-04-02,1690,professional,1,N


Station 337


date                2026-04-01  2026-04-02  2026-04-03  2026-04-04  2026-04-05  2026-04-06  2026-04-07  2026-04-08  2026-04-09  2026-04-10  2026-04-11  2026-04-12  2026-04-13  2026-04-14  \
shift staff_level                                                                                                                                                                            
F     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   

date                2026-04-15  2026-04-16  2026-04-17  2026-04-18  2026-04-19  2026-04-20  2026-04-21  2026-04-22  2026-04-23  2026-04-24  2026-04-25  2026-04-26  2026-04-27  2026-04-28  \
shift staff_level                                                                                                                                                                            
F     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   

date                2026-04-29  2026-04-30  
shift staff_level                           
F     professional           1           1  
      trainee                1           1  
N     professional           1           1  
S     professional           1           1  
      trainee                1           1

In [7]:
accounts = table(dataset.monthly_work_accounts, drop={"credit_details"})
accounts["target_h"] = (accounts["target_minutes"] / 60).round(1)
accounts["open_minutes"] = accounts["target_minutes"] - accounts["credited_minutes"]   # noch zu verplanen
display(accounts)

# Einzelne Gutschriften (Urlaub, Schule, ...)
credits = pd.DataFrame([{"employee_id": a.employee_id, **c.model_dump(mode="json")}
                        for a in dataset.monthly_work_accounts for c in a.credit_details])
display(credits)


,employee_id,target_minutes,actual_minutes,credited_minutes,target_h,open_minutes
0,918,6000,None,0,100.0,6000
1,835,1200,None,0,20.0,1200
2,7006,9240,None,0,154.0,9240
3,843,4080,None,0,68.0,4080
4,6318,9240,None,0,154.0,9240
5,1167,9240,None,0,154.0,9240
6,839,4620,None,0,77.0,4620
7,8046,9240,None,0,154.0,9240
8,8047,9240,None,0,154.0,9240
9,8048,9240,None,0,154.0,9240


""


In [8]:
context = table(dataset.context.duties)
if not context.empty:
    context["shift"] = context["shift_id"].map(code)
    display(context.pivot_table(index="employee_id", columns="date", values="shift",
                                aggfunc="first", fill_value=""))
display(table(dataset.context.availability))


date,2026-03-27,2026-03-28,2026-03-29,2026-03-30,2026-03-31
employee_id,,,,,
839,F,,,F,F
843,,,,,S
918,S,,,,
1167,F,S,S,S,S
5844,,F,F,F,F
6259,N,,,,F
6318,F,,,F,F
6724,,,,,N
7006,S,S,,S,S


""


In [9]:
display(pd.Series(POLICY.model_dump(), name="Wert").to_frame())


,Wert
balance_tolerance_minutes,460
min_rest_minutes,660
max_consecutive_nights,3
night_recovery_minutes,2880
max_daily_work_minutes,600
average_daily_work_minutes,480
short_break_after_minutes,360
short_break_minutes,30
long_break_after_minutes,540
long_break_minutes,45


In [10]:
class GurobiSolverService:
    def __init__(self, settings: Settings) -> None:
        self._settings = settings        

    def solve(self, dataset: SchedulingDataset, timeout: float | None = None) -> Solution:
        timeout = timeout if timeout is not None else self._settings.solver_max_time_seconds
        with gp.Env() as env, gp.Model("schedule", env=env) as schedule:
            # Sets and Indices
            U = []
            for unit in dataset.planning_units:
                U.append(unit.planning_unit_id)
            L = [level.value for level in StaffLevel]
            stations_of = {
                e.employee_id: sorted({
                    m.planning_unit_id
                    for m in dataset.planning_unit_memberships
                    if m.employee_id == e.employee_id and m.planning_unit_id in dataset.station_ids
                })
                for e in dataset.employees
            }
            I = []
            I_l = {l:[] for l in L}
            for employee in dataset.employees:
                I.append(employee.employee_id)
                I_l[employee.staff_level].append(employee.employee_id)
            S = []
            for shift in dataset.shifts:
                S.append(shift.shift_id)
            S_n = [s.shift_id for s in dataset.shifts if s.type == ShiftType.NIGHT]
            D = dataset.planning_month.dates
            # Parameters
            R_min = {}
            for s in S:
                for d in D:
                    for u in U:
                        for l in L:
                            R_min[s, d, u, l] = 0
            for entry in dataset.demand_requirements:
                R_min[entry.shift_id,entry.date,entry.planning_unit_id,entry.staff_level] = entry.required_count
            # Decision variables
            assignment = schedule.addVars(I, S, D, U, vtype=GRB.BINARY,name="assignment")
            gap = schedule.addVars(S, D, U, L, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="gap")
            n_beg = schedule.addVars(I, D, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_beg")
            n_end = schedule.addVars(I, D, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_end")
            # Objective functions
            schedule.setObjectiveN(gap.sum(), index=0, priority=1, name="Min_Gaps")
            schedule.setObjectiveN(assignment.sum(), index=1, priority=0, name="Min_Assignments")
            schedule.ModelSense = GRB.MINIMIZE
            # Constraints
            # one shift per day
            schedule.addConstrs((assignment.sum(i, '*', d, '*') <= 1 for l in L for i in I for d in D));
            # only assign on allowed units
            for i in I:
                for u in U:
                    if u not in stations_of[i]:
                        for s in S:
                            for d in D:
                                assignment[i,s,d,u].ub = 0
            # min demand
            for l in L:
                schedule.addConstrs((assignment.sum(I_l[l], s, d, u) + gap[s, d, u, l] >= R_min[s, d, u, l] for s in S for d in D for u in U));
            # max consecutive nights
            for i in I:
                for d in D[:-POLICY.max_consecutive_nights]:
                    window = [d+timedelta(days=i) for i in range(POLICY.max_consecutive_nights + 1)]
                    schedule.addConstr(assignment.sum(i,S_n,window,'*')<= POLICY.max_consecutive_nights)
            # define beginning of nightshiftblock
            schedule.addConstrs((n_beg[i,d] >= assignment.sum(i,S_n,d,'*') - assignment.sum(i,S_n,d-timedelta(days=1),'*') for i in I for d in D[1:]));
            schedule.addConstrs((n_beg[i,d] <= assignment.sum(i,S_n,d,'*') for i in I for d in D))
            schedule.addConstrs((n_beg[i,d] <= 1 - assignment.sum(i,S_n,d-timedelta(days=1),'*') for i in I for d in D[1:]));
            # define end of nightshiftblock
            schedule.addConstrs((n_end[i,d] >= assignment.sum(i,S_n,d,'*') - assignment.sum(i,S_n,d+timedelta(days=1),'*') for i in I for d in D[:-1]));
            schedule.addConstrs((n_end[i,d] <= assignment.sum(i,S_n,d,'*') for i in I for d in D));
            schedule.addConstrs((n_end[i,d] <= 1 - assignment.sum(i,S_n,d+timedelta(days=1),'*') for i in I for d in D[:-1]));
            # no nightshift block starts and ends on the same day
            schedule.addConstrs(n_beg[i,d]+n_end[i,d]<=1 for i in I for d in D);
            # ensure recovery days after nightshift block
            L_n = POLICY.night_recovery_minutes//1440
            for i in I:
                for d in D[:-L_n]:
                    window = [d+timedelta(days=i) for i in range(1,L_n+1)]
                    schedule.addConstr(n_end[i, d] <= 1 - assignment.sum(i, '*', window, '*'))
            # same shifts on weekends
            schedule.addConstrs((assignment.sum(i, s, d, '*') == assignment.sum(i, s, d+timedelta(days=1),'*') for i in I for s in S for d in D_s));
            # solve
            schedule.setParam("TimeLimit", timeout)
            schedule.optimize()
            # construct solution
            assignments = tuple(
                Assignment(employee_id=i, shift_id=s, date=d, planning_unit_id=u, staff_level=l)
                for l in L for i in I_l[l] for s in S for d in D for u in U
                if assignment[i, s, d, u].X > 0.5
            )
            gaps = tuple(
                Gap(shift_id=s, date=d, planning_unit_id=u, staff_level=l, missing_count=int(var.X))
                for (s,d,u,l), var in gap.items()
                if int(var.X) > 0
            )
            stages = []
            for i in range(schedule.NumObj):
                schedule.Params.ObjNumber = i
                stages.append(StageReport(
                    name=schedule.ObjNName,
                    status=SolutionStatus.OPTIMAL if schedule.ObjPassNStatus==2 else SolutionStatus.FEASIBLE,
                    value=int(schedule.ObjNVal),
                    best_bound=int(schedule.ObjPassNObjBound)))
            stages = tuple(stages)
            if schedule.Status == GRB.OPTIMAL:
                status = SolutionStatus.OPTIMAL
            elif schedule.Status in (GRB.INFEASIBLE, GRB.INF_OR_UNBD):
                status = SolutionStatus.INFEASIBLE
            elif schedule.SolCount > 0:
                status = SolutionStatus.FEASIBLE
            else:
                status = SolutionStatus.UNKNOWN

            found = status in (SolutionStatus.OPTIMAL, SolutionStatus.FEASIBLE)
            return Solution(
                status=status,
                configuration=RunConfiguration(policy=POLICY, timeout_seconds=schedule.Params.TimeLimit,
                                            search_workers=None, random_seed=None),
                wall_time_seconds=schedule.Runtime,
                assignments=assignments if found else (),
                gaps=gaps if found else (),
                stages=stages if found else (),
                check=check_schedule(dataset, assignments, gaps) if found else None,
            )


In [11]:
U = []
for unit in dataset.planning_units:
    U.append(unit.planning_unit_id)
L = [level.value for level in StaffLevel]
stations_of = {
    e.employee_id: sorted({
        m.planning_unit_id
        for m in dataset.planning_unit_memberships
        if m.employee_id == e.employee_id and m.planning_unit_id in dataset.station_ids
    })
    for e in dataset.employees
}

In [ ]:
I = []
I_l = {l:[] for l in L}
for employee in dataset.employees:
    I.append(employee.employee_id)
    I_l[employee.staff_level].append(employee.employee_id)
S = []
for shift in dataset.shifts:
    S.append(shift.shift_id)
S_n = [s.shift_id for s in dataset.shifts if s.type == ShiftType.NIGHT]
S_e = [s.shift_id for s in dataset.shifts if s.type == ShiftType.EARLY]
D = dataset.planning_month.dates
D_s = [date for date in dataset.planning_month.dates if date.weekday() == 5]

In [13]:
# D_prev = [dataset.context.covered_from + timedelta(days=i) for i in range((dataset.planning_month.dates[0] - dataset.context.covered_from).days)]
# D_e = [date.day for date in dataset.planning_month.dates if date.weekday() >= 5]

In [14]:
R_min = {}
for s in S:
    for d in D:
        for u in U:
            for l in L:
                R_min[s, d, u, l] = 0
for entry in dataset.demand_requirements:
    R_min[entry.shift_id,entry.date,entry.planning_unit_id,entry.staff_level] = entry.required_count

In [96]:
schedule = gp.Model("schedule")

In [97]:
assignment = schedule.addVars(I, S, D, U, vtype=GRB.BINARY,name="assignment")
gap = schedule.addVars(S, D, U, L, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="gap")
n_beg = schedule.addVars(I, D, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_beg")
n_end = schedule.addVars(I, D, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_end")

In [98]:
schedule.setObjectiveN(gap.sum(), index=0, priority=1, name="Min_Gaps")
schedule.setObjectiveN(assignment.sum(), index=1, priority=0, name="Min_Assignments")
schedule.ModelSense = GRB.MINIMIZE

In [99]:
# one shift per day
schedule.addConstrs((assignment.sum(i, '*', d, '*') <= 1 for l in L for i in I for d in D));

In [100]:
# assign only on allowed units
for i in I:
    for u in U:
        if u not in stations_of[i]:
            for s in S:
                for d in D:
                    assignment[i,s,d,u].ub = 0

In [101]:
# min demand
for l in L:
    schedule.addConstrs((assignment.sum(I_l[l], s, d, u) + gap[s, d, u, l] >= R_min[s, d, u, l] for s in S for d in D for u in U));

In [102]:
# define beginning of nightshiftblock
schedule.addConstrs((n_beg[i,d] >= assignment.sum(i,S_n,d,'*') - assignment.sum(i,S_n,d-timedelta(days=1),'*') for i in I for d in D[1:]));
schedule.addConstrs((n_beg[i,d] <= assignment.sum(i,S_n,d,'*') for i in I for d in D))
schedule.addConstrs((n_beg[i,d] <= 1 - assignment.sum(i,S_n,d-timedelta(days=1),'*') for i in I for d in D[1:]));

In [103]:
# define end of nightshiftblock
schedule.addConstrs((n_end[i,d] >= assignment.sum(i,S_n,d,'*') - assignment.sum(i,S_n,d+timedelta(days=1),'*') for i in I for d in D[:-1]));
schedule.addConstrs((n_end[i,d] <= assignment.sum(i,S_n,d,'*') for i in I for d in D));
schedule.addConstrs((n_end[i,d] <= 1 - assignment.sum(i,S_n,d+timedelta(days=1),'*') for i in I for d in D[:-1]));

In [104]:
# max consecutive nights
for i in I:
    for d in D[:-POLICY.max_consecutive_nights]:
        window = [d+timedelta(days=i) for i in range(POLICY.max_consecutive_nights + 1)]
        schedule.addConstr(assignment.sum(i,S_n,window,'*')<= POLICY.max_consecutive_nights)

In [105]:
# ensure recovery days after nightshift block
L_n = POLICY.night_recovery_minutes//1440
for i in I:
    for d in D[:-L_n]:
        for j in range(1,L_n+1):
            schedule.addConstr(n_end[i, d] + assignment.sum(i, '*', d + timedelta(days=j), '*') <= 1)
        # no early shift after recovery days
        schedule.addConstr((n_end[i, d] + assignment.sum(i, S_e, d + timedelta(days=L_n+1), '*') <= 1))


In [106]:
# no nightshift block starts and ends on the same day
schedule.addConstrs(n_beg[i,d]+n_end[i,d]<=1 for i in I for d in D);

In [107]:
# same shifts on weekends
schedule.addConstrs((assignment.sum(i, s, d, '*') == assignment.sum(i, s, d+timedelta(days=1),'*') for i in I for s in S for d in D_s));

___

In [108]:
schedule.setParam("TimeLimit", 300)
schedule.optimize()

Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: Snapdragon(R) X Elite - X1E78100 - Qualcomm(R) Oryon(TM) CPU, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 10500 rows, 9840 columns and 81340 nonzeros (Min)
Model fingerprint: 0x441fb333
Model has 1440 linear objective coefficients
Variable types: 2640 continuous, 7200 integer (7200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]


---------------------------------------------------------------------------
Multi-objectives: starting optimization (min) with 2 objectives... 
---------------------------------------------------------------------------

Multi-objectives: applying initial presolve...
-----------------

In [109]:
assignments = tuple(
    Assignment(employee_id=i, shift_id=s, date=d, planning_unit_id=u, staff_level=l)
    for l in L for i in I_l[l] for s in S for d in D for u in U
    if assignment[i, s, d, u].X > 0.5
)

In [110]:
gaps = tuple(
    Gap(shift_id=s, date=d, planning_unit_id=u, staff_level=l, missing_count=int(var.X))
    for (s,d,u,l), var in gap.items()
    if int(var.X) > 0
)

In [111]:
stages = []
for i in range(schedule.NumObj):
    schedule.Params.ObjNumber = i
    stages.append(StageReport(
        name=schedule.ObjNName,
        status=SolutionStatus.OPTIMAL if schedule.ObjPassNStatus==2 else SolutionStatus.FEASIBLE,
        value=int(schedule.ObjNVal),
        best_bound=int(schedule.ObjPassNObjBound)))
stages = tuple(stages)

In [112]:
if schedule.Status == GRB.OPTIMAL:
    status = SolutionStatus.OPTIMAL
elif schedule.Status in (GRB.INFEASIBLE, GRB.INF_OR_UNBD):
    status = SolutionStatus.INFEASIBLE
elif schedule.SolCount > 0:
    status = SolutionStatus.FEASIBLE
else:
    status = SolutionStatus.UNKNOWN

found = status in (SolutionStatus.OPTIMAL, SolutionStatus.FEASIBLE)
solution = Solution(
    status=status,
    configuration=RunConfiguration(policy=POLICY, timeout_seconds=schedule.Params.TimeLimit,
                                   search_workers=None, random_seed=None),
    wall_time_seconds=schedule.Runtime,
    assignments=assignments if found else (),
    gaps=gaps if found else (),
    stages=stages if found else (),
    check=check_schedule(dataset, assignments, gaps) if found else None,
)

In [113]:
findings = [f for f in solution.check.findings]

In [114]:
len(findings)

48

In [115]:
findings

[Finding(rule=<Rule.MONTHLY_BALANCE: 'monthly_balance'>, message='Balance +2880 minutes (target 6000, credits 0) is outside ±460.', employee_id=918, date=None, planning_unit_id=None, shift_id=None),
 Finding(rule=<Rule.MONTHLY_BALANCE: 'monthly_balance'>, message='Balance -780 minutes (target 1200, credits 0) is outside ±460.', employee_id=835, date=None, planning_unit_id=None, shift_id=None),
 Finding(rule=<Rule.MONTHLY_BALANCE: 'monthly_balance'>, message='Balance -8385 minutes (target 9240, credits 0) is outside ±460.', employee_id=7006, date=None, planning_unit_id=None, shift_id=None),
 Finding(rule=<Rule.MONTHLY_BALANCE: 'monthly_balance'>, message='Balance -4080 minutes (target 4080, credits 0) is outside ±460.', employee_id=843, date=None, planning_unit_id=None, shift_id=None),
 Finding(rule=<Rule.MONTHLY_BALANCE: 'monthly_balance'>, message='Balance -8805 minutes (target 9240, credits 0) is outside ±460.', employee_id=6318, date=None, planning_unit_id=None, shift_id=None),
 Fin

In [87]:
import platform
from hashlib import sha256
from pathlib import Path

import pydantic
from app.solver.bundle import InputReference, RuntimeVersions, ScheduleInput, ScheduleResult, to_json

out = Path("export")
out.mkdir(exist_ok=True)

input_json = to_json(ScheduleInput.of(dataset))
result = ScheduleResult(
    format_version=3,
    planning_month=dataset.planning_month,
    input=InputReference(file="input.json", sha256=sha256(input_json).hexdigest()),
    solution=solution,
    versions=RuntimeVersions(python=platform.python_version(), ortools="gurobi", pydantic=pydantic.VERSION),
)
(out / "input.json").write_bytes(input_json)
(out / "result.json").write_bytes(to_json(result))


40945

In [88]:
for d in D:
    for i in I:
        if n_end[i,d].X > .5:
            print(i,d)

918 2026-04-03
7490 2026-04-06
918 2026-04-09
8050 2026-04-12
6724 2026-04-14
918 2026-04-16
7006 2026-04-19
918 2026-04-22
6318 2026-04-24
1167 2026-04-27
